# Model checksum verification 

**What this notebook is:** my own study notes explaining a small feature I added to the production backend — why I added it, the real code, and how I checked it works. I wrote it in plain language so I (or a teammate) can follow it later.

**Where the feature lives:** `software_side/walkbuddy_reactNative/backend/`



---

## 1. The task and what I found

My task was **"Optimize and validate production backend integration"** — get the approved obstacle-detection model properly integrated into the production backend (`backend/main.py`) and make sure it actually works.

Before writing any code, I validated the existing integration end-to-end:

- the model loads through `backend/main.py` at startup,
- `GET /ml/model-info` reports the model's SHA-256 and its class list,
- `POST /ml/navigate` returns real predictions,
- the backend test suite passes (`165 passed` under `tests/`).

So the integration itself already worked. But while reading the code I found a **gap**:

> The backend **recorded and reported** the loaded model's SHA-256 (it computed the hash at startup and exposed it on `/ml/model-info`), but it **never verified** that SHA-256 against an expected/approved value.

That means if someone dropped a **wrong or corrupted `best.pt`** into the model folder, the backend would load it **silently** — the hash shown on `/ml/model-info` would just change, and nothing would flag it.

I also searched the repo to make sure I wasn't about to duplicate something that already existed. The only checksum-*verification* logic lived on the ML side (dataset-release checks and model-registry promotion gates) — **none of it ran in the backend at startup**. The backend only ever *computed* the hash. So adding a startup verification was genuinely new work, not a duplicate.

The code below is the pre-existing helper that already computed the hash (this existed **before** my change). Notice it only calculates and returns the hash — it never compares it to anything:

In [ ]:
# --- pre-existing code in ml_runtime/model_info.py (existed BEFORE my change) ---
# It streams the file in chunks and returns the SHA-256. That's all it did:
# compute + return. Nothing here compares the hash to an approved value.

CHECKSUM_CHUNK_SIZE = 1024 * 1024


def calculate_sha256(path: Path) -> str:
    """Return the SHA-256 of a local artifact without modifying it."""
    digest = hashlib.sha256()
    with path.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(CHECKSUM_CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 2. What I added — overview

I added a **startup checksum verification**. The design goals I stuck to:

- **Pure addition.** It doesn't change `/vision`, `/ws/vision`, or `/ml/navigate`, and it does **not** replace the model.
- **Compare, don't just record.** At startup it now compares the loaded model's SHA-256 to an *expected approved value*.
- **Surface the result.** It adds a new field `checksum_verified` to `GET /ml/model-info`.
- **Never crash.** On a mismatch it logs a clear error and records the result, but the app keeps running and the model stays usable — the same "report, don't enforce" style the existing code already used for model status.
- **Independent of the taxonomy check.** There was already a `taxonomy_compatible` field (does the model's class list match the approved taxonomy?). My `checksum_verified` is a *separate* signal: one is about **file integrity**, the other about **class labels**. I deliberately did not couple them.

`checksum_verified` has three possible values:

| value | meaning |
| --- | --- |
| `true` | the loaded model's SHA-256 matched the expected value |
| `false` | it did **not** match (logged; the app still runs) |
| `null` | "not checked" — no expected value was configured |

The core of the idea is one tiny pure function. Here it is copied out as a **runnable demo** so I can see all three outcomes at once (the real version lives in `model_info.py`, shown in section 3):

In [ ]:
# Runnable demo of the core idea. This is the exact body of the helper I added
# in ml_runtime/model_info.py, copied here so this cell runs on its own.
def verify_checksum(lineage_sha, expected_sha):
    if not lineage_sha or not expected_sha:
        return None            # "not checked" (either value missing)
    return lineage_sha.strip().lower() == expected_sha.strip().lower()


sha = "198df54da4f6aa071b342bee77b100e78f243df785b325ec364036e106572238"

print("match      ->", verify_checksum(sha, sha))       # True
print("mismatch   ->", verify_checksum(sha, "0" * 64))  # False
print("not set    ->", verify_checksum(sha, None))       # None

## 3. The changed files

I'll go file by file: **why** I touched it and **what** I changed, then the real code.

### 3a. `ml_runtime/model_info.py`

**Why:** this is where the model's "lineage" (safe metadata about the loaded model) is defined and where the SHA-256 was already computed. It's the natural home for the comparison logic and the new field.

**What I changed:**

1. Added a pure `verify_checksum()` helper (no side effects, never raises).
2. Added a `checksum_verified: bool | None = None` field on the `ModelLineage` dataclass, **mirroring** the existing `taxonomy_compatible` field, and included it in `as_dict()` so it shows up on `/ml/model-info`.

Because the field has a default of `None`, every existing way of building a lineage keeps working unchanged and just reports "not checked" until the verification runs.

In [ ]:
# --- ml_runtime/model_info.py (real code) ---

@dataclass(frozen=True)
class ModelLineage:
    """The intentionally limited model details safe to expose operationally."""

    loaded: bool
    filename: str
    sha256: str | None
    size_bytes: int | None
    num_classes: int | None
    classes: list[str]
    load_duration_ms: float
    loaded_at: str
    runtime: dict[str, Any]
    failure_category: str | None = None
    taxonomy_compatible: bool | None = None
    checksum_verified: bool | None = None   # <-- my new field (mirrors taxonomy_compatible)

    def as_dict(self) -> dict[str, Any]:
        """Return a copy suitable for the operational model-info endpoint."""
        return {
            "loaded": self.loaded,
            "filename": self.filename,
            "sha256": self.sha256,
            "size_bytes": self.size_bytes,
            "num_classes": self.num_classes,
            "classes": list(self.classes),
            "taxonomy_compatible": self.taxonomy_compatible,
            "checksum_verified": self.checksum_verified,   # <-- surfaced on /ml/model-info
            "load_duration_ms": self.load_duration_ms,
            "loaded_at": self.loaded_at,
            "runtime": dict(self.runtime),
            "failure_category": self.failure_category,
        }


def verify_checksum(lineage_sha: str | None, expected_sha: str | None) -> bool | None:
    """Compare a loaded artifact's SHA-256 to an expected/approved value.

    Returns ``True`` or ``False`` only when both values are present; returns
    ``None`` ("not checked") when either side is missing — i.e. no expected
    value was configured, or the model exposes no checksum. Comparison ignores
    surrounding whitespace and hex letter-casing.

    This is a pure, side-effect-free helper. It never raises and is deliberately
    independent of :func:`is_taxonomy_compatible`; checksum integrity and class
    taxonomy are reported as two separate, orthogonal signals.
    """
    if not lineage_sha or not expected_sha:
        return None
    return lineage_sha.strip().lower() == expected_sha.strip().lower()

### 3b. `ml_runtime/state.py`

**Why:** `MLRuntimeState` is the single object that owns the model lineage and hands it to `/ml/model-info`. The `ModelLineage` is a *frozen* dataclass (immutable), and it's already stored by the time I know the verification result. So I needed a small, thread-safe way to record the result onto the lineage that's already there.

**What I changed:**

1. Added `set_checksum_verification()`. It uses `dataclasses.replace` to produce an updated copy of the stored lineage with `checksum_verified` set. It only *annotates* an already-captured lineage — it never marks a loaded model as unloaded, and it stays independent of taxonomy. A mismatch may optionally attach a `failure_category` for operators, but the model stays `loaded`.
2. Added the `checksum_verified` key to the "not initialized" payload (the response returned before any model has loaded), so the field is **always present** on `/ml/model-info`.

In [ ]:
# --- ml_runtime/state.py (real code) ---
# (from dataclasses import replace  # added at the top of the file)

    def set_checksum_verification(
        self, checksum_verified: bool | None, *, failure_category: str | None = None
    ) -> None:
        """Record the startup checksum-verification result on the active lineage.

        This is a pure annotation of an already-captured lineage: it never marks
        a loaded model as unloaded and stays independent of taxonomy reporting.
        A mismatch may optionally attach a ``failure_category`` for operators,
        but the model remains ``loaded`` and usable regardless.
        """
        with self._model_lock:
            if self._model_lineage is None:
                return
            updates: dict[str, Any] = {"checksum_verified": checksum_verified}
            if failure_category is not None:
                updates["failure_category"] = failure_category
            self._model_lineage = replace(self._model_lineage, **updates)

    # ... and inside model_info(), the pre-startup payload now always includes
    # the new key so /ml/model-info never omits it:
    #
    #     "taxonomy_compatible": None,
    #     "checksum_verified": None,   # <-- added
    #     "load_duration_ms": None,

### 3c. `main.py`

**Why:** `main.py` is where the app starts up and loads the model (the `lifespan` function). This is the one place that knows the model just loaded and can run the check once at boot. It's also the right layer to decide *where the expected value comes from*.

**What I changed:**

1. Added a small resolver, `_resolve_expected_model_sha256()`, that finds the expected SHA in a clear priority order (explained in section 4): the environment variable first, then a committed baseline JSON, else `None`. The expected value is **not hardcoded** anywhere in the code.
2. In the lifespan's successful-load branch, right after the lineage is recorded, I compute `verify_checksum(...)` and then:
   - **mismatch (`False`)** → log a clear `error` and record `checksum_verified=False` (plus `failure_category="model_checksum_mismatch"`), but **do not raise**;
   - **match (`True`)** → log success and record `True`;
   - **not configured (`None`)** → log an info line and record `None`.

The whole block sits *after* the model is already loaded and recorded, so nothing about it can stop the model from being usable.

In [ ]:
# --- main.py: the expected-SHA resolver (real code) ---

# Expected/approved SHA-256 for the shared YOLO artifact. This is intentionally
# NOT hardcoded in code: it is sourced (in priority order) from an environment
# variable, then falling back to the machine-readable approved record that the
# ML side already maintains. When neither is available the check is skipped
# (reported as "not checked") rather than failing startup.
_EXPECTED_MODEL_SHA256_ENV = "WALKBUDDY_EXPECTED_MODEL_SHA256"
_BASELINE_SHA_PATH = PROJECT_ROOT / "ML_side/evaluation/baselines/historical_7class_baseline.json"


def _expected_sha_from_baseline() -> str | None:
    """Read the approved SHA-256 from the committed baseline record, if present.

    Weights are never committed to git, but the ML side already records the
    approved artifact's SHA-256 in this versioned JSON, so it is a safe single
    source of truth for the fallback. Any read/parse problem is treated as
    "unavailable" and must never break startup.
    """
    try:
        with _BASELINE_SHA_PATH.open("r", encoding="utf-8") as baseline_file:
            record = json.load(baseline_file)
        sha256 = record.get("model", {}).get("sha256")
    except (OSError, ValueError, AttributeError):
        return None
    if isinstance(sha256, str) and sha256.strip():
        return sha256.strip()
    return None


def _resolve_expected_model_sha256() -> str | None:
    """Resolve the expected SHA-256: env var first, then baseline, else None."""
    env_value = os.environ.get(_EXPECTED_MODEL_SHA256_ENV, "").strip()
    if env_value:
        return env_value
    return _expected_sha_from_baseline()

In [ ]:
# --- main.py: the verification step inside lifespan() (real code) ---
# This lives in the "model loaded successfully" branch, right AFTER the lineage
# has been recorded, so the model is already usable before this runs.

        else:
            app.state.ml_runtime.set_model_lineage(lineage)

            # --- verify checksum against the approved value (never crash) ---
            # This is a pure addition: it only annotates the recorded lineage and
            # is independent of the taxonomy compatibility signal. A mismatch is
            # logged and surfaced via /ml/model-info, but the model stays loaded
            # and /vision, /ws/vision, and /ml/navigate are unaffected.
            expected_sha256 = _resolve_expected_model_sha256()
            checksum_verified = verify_checksum(lineage.sha256, expected_sha256)
            if checksum_verified is False:
                logger.error(
                    "❌ Model checksum verification FAILED: the loaded best.pt "
                    "SHA-256 does not match the expected approved value. The "
                    "model remains loaded and usable, but its integrity is "
                    "UNVERIFIED — investigate the artifact before relying on it."
                )
                app.state.ml_runtime.set_checksum_verification(
                    False, failure_category="model_checksum_mismatch"
                )
            elif checksum_verified is True:
                logger.info("✅ Model checksum verified against expected SHA-256")
                app.state.ml_runtime.set_checksum_verification(True)
            else:
                logger.info(
                    "ℹ️ Model checksum not verified: no expected SHA-256 "
                    "configured (set %s or provide the baseline record)",
                    _EXPECTED_MODEL_SHA256_ENV,
                )
                app.state.ml_runtime.set_checksum_verification(None)
        logger.info("✅ YOLO ready")

### 3d. The tests I added in `tests/test_ml_runtime.py`

**Why:** I wanted the three behaviours locked down so they can't silently regress: a match reports `True`, a mismatch reports `False` **and the app still runs**, and "not configured" reports `None`. I also wanted to prove the resolver priority (env → baseline → None) and that the real `lifespan` startup does the right thing.

**What I added** (real code below):

- **Pure helper tests** — `verify_checksum` for match / mismatch / not-configured.
- **Resolver test** — env var wins, else baseline JSON, else `None`.
- **Real-lifespan tests** — a small `_FakeYolo` and a helper that actually runs `main.lifespan(...)`, then three tests: matching SHA → `checksum_verified True`; wrong SHA → `False` + `failure_category` **and the app still starts** (`yolo` is not `None`); nothing configured → `None`.

(While wiring these up I also stubbed `routers.ml_inference` in the shared `main_module` test fixture so the lifespan tests run in isolation, not only as part of the full suite.)

In [ ]:
# --- tests/test_ml_runtime.py (real code, selected tests) ---

def test_verify_checksum_matches_returns_true() -> None:
    sha = "198df54da4f6aa071b342bee77b100e78f243df785b325ec364036e106572238"
    assert verify_checksum(sha, sha) is True
    # Whitespace and hex letter-casing must not defeat a genuine match.
    assert verify_checksum(sha, f"  {sha.upper()}  ") is True


def test_verify_checksum_mismatch_returns_false() -> None:
    assert verify_checksum("a" * 64, "b" * 64) is False


def test_verify_checksum_not_configured_returns_none() -> None:
    # "not checked" whenever either side is missing — never a spurious False.
    assert verify_checksum("a" * 64, None) is None
    assert verify_checksum("a" * 64, "") is None
    assert verify_checksum(None, "a" * 64) is None
    assert verify_checksum(None, None) is None


def test_resolve_expected_sha_prefers_env_then_baseline_then_none(
    main_module: ModuleType, monkeypatch: pytest.MonkeyPatch, tmp_path: Path
) -> None:
    # 1. Environment variable wins.
    monkeypatch.setenv("WALKBUDDY_EXPECTED_MODEL_SHA256", "a" * 64)
    assert main_module._resolve_expected_model_sha256() == "a" * 64

    # 2. Without the env var, fall back to the committed baseline record.
    monkeypatch.delenv("WALKBUDDY_EXPECTED_MODEL_SHA256", raising=False)
    baseline = tmp_path / "baseline.json"
    baseline.write_text(json.dumps({"model": {"sha256": "b" * 64}}))
    monkeypatch.setattr(main_module, "_BASELINE_SHA_PATH", baseline)
    assert main_module._resolve_expected_model_sha256() == "b" * 64

    # 3. With neither, resolution is None (verification is skipped, not failed).
    monkeypatch.setattr(main_module, "_BASELINE_SHA_PATH", tmp_path / "missing.json")
    assert main_module._resolve_expected_model_sha256() is None

In [ ]:
# --- tests/test_ml_runtime.py: the real-lifespan tests (real code) ---

class _FakeYolo:
    """A stand-in loaded model exposing the legacy 7-class names mapping."""

    def __init__(self, _path: str) -> None:
        self.names = dict(FakeModel.names)


def _run_lifespan_with_loaded_model(
    main_module: ModuleType,
    monkeypatch: pytest.MonkeyPatch,
    artifact: Path,
) -> dict[str, object]:
    """Drive the real lifespan with a successfully 'loaded' model artifact."""
    artifact.write_bytes(b"actual-weights")
    main_module.YOLO_MODEL_PATH = artifact
    main_module.init_database = lambda: None
    monkeypatch.setattr(main_module, "YOLO", _FakeYolo)
    monkeypatch.setattr(main_module, "_cleanup_sessions_loop", lambda: None)
    monkeypatch.setattr(main_module.asyncio, "create_task", lambda _coroutine: None)

    async def start() -> dict[str, object]:
        async with main_module.lifespan(main_module.app):
            return {
                "model_info": main_module.app.state.ml_runtime.model_info(),
                "yolo": main_module.app.state.yolo,
            }

    return asyncio.run(start())


def test_lifespan_verifies_matching_checksum(
    main_module: ModuleType, monkeypatch: pytest.MonkeyPatch, tmp_path: Path
) -> None:
    artifact = tmp_path / "best.pt"
    # Point the expected value at the artifact's real SHA so it matches.
    artifact.write_bytes(b"actual-weights")
    monkeypatch.setenv(
        "WALKBUDDY_EXPECTED_MODEL_SHA256", calculate_sha256(artifact)
    )

    result = _run_lifespan_with_loaded_model(main_module, monkeypatch, artifact)

    assert result["yolo"] is not None
    assert result["model_info"]["loaded"] is True
    assert result["model_info"]["checksum_verified"] is True
    assert result["model_info"]["failure_category"] is None


def test_lifespan_records_checksum_mismatch_without_crashing(
    main_module: ModuleType, monkeypatch: pytest.MonkeyPatch, tmp_path: Path
) -> None:
    artifact = tmp_path / "best.pt"
    # A deliberately wrong expected value forces the mismatch (warning) path.
    monkeypatch.setenv("WALKBUDDY_EXPECTED_MODEL_SHA256", "0" * 64)

    result = _run_lifespan_with_loaded_model(main_module, monkeypatch, artifact)

    # Startup completed and the model remains loaded and usable.
    assert result["yolo"] is not None
    assert result["model_info"]["loaded"] is True
    assert result["model_info"]["checksum_verified"] is False
    assert result["model_info"]["failure_category"] == "model_checksum_mismatch"


def test_lifespan_skips_checksum_when_not_configured(
    main_module: ModuleType, monkeypatch: pytest.MonkeyPatch, tmp_path: Path
) -> None:
    artifact = tmp_path / "best.pt"
    monkeypatch.delenv("WALKBUDDY_EXPECTED_MODEL_SHA256", raising=False)
    # No baseline record available either -> nothing to check.
    monkeypatch.setattr(main_module, "_BASELINE_SHA_PATH", tmp_path / "missing.json")

    result = _run_lifespan_with_loaded_model(main_module, monkeypatch, artifact)

    assert result["yolo"] is not None
    assert result["model_info"]["loaded"] is True
    assert result["model_info"]["checksum_verified"] is None
    # "Not checked" is not a failure.
    assert result["model_info"]["failure_category"] is None

## 4. How I sourced the expected value

The tricky part was: **where does the "expected" SHA-256 come from?** The model weights (`best.pt`) are **not committed to git** (they live in SharePoint and are downloaded locally), so I can't just read the approved hash off the weights file in the repo, and I didn't want to paste a hash literal into the code where it could rot.

I settled on a **priority order**:

1. **Environment variable `WALKBUDDY_EXPECTED_MODEL_SHA256`** — first choice. This is the right knob for deployment: whoever deploys pins the approved hash for that environment. It also matches the backend's existing env-var style (`WALKBUDDY_MODEL_DIR`, `WALKBUDDY_API_KEY`, etc.).
2. **Committed baseline JSON** — fallback. Even though the *weights* aren't in git, the ML side already records the approved artifact's SHA-256 in a versioned file: `ML_side/evaluation/baselines/historical_7class_baseline.json`. That's a real, reviewed source of truth I can read without hardcoding anything.
3. **`None`** — if neither is set, verification is simply **skipped** (`checksum_verified: null`), not failed. Skipping when unconfigured is safer than crashing.

The relevant part of that committed baseline record looks like this (the `model.sha256` is what the fallback reads):

In [ ]:
# --- excerpt of ML_side/evaluation/baselines/historical_7class_baseline.json ---
# (the fallback reads record["model"]["sha256"])
{
  "model": {
    "filename": "best.pt",
    "file_size_bytes": 6244458,
    "sha256": "198df54da4f6aa071b342bee77b100e78f243df785b325ec364036e106572238",
    "class_count": 7,
    "ordered_class_names": [
      "book", "books", "monitor", "office-chair", "whiteboard", "table", "tv"
    ]
  }
}

## 5. How I verified it live on my own machine

Tests are good, but I also wanted to see it run for real against the actual model. Here's what I did and what I observed (the shell cell below shows the commands and the real output/log lines).

- **Tests:** `pytest tests/` → **165 passed, 0 failed**. (The `sklearn` failure that can appear elsewhere is under `predictive_path/`, not `tests/`, so it doesn't show up in the backend test suite.)
- **Normal startup (match):** I started the backend with the real model and *without* setting the env var, so the expected SHA came from the **baseline fallback**. The log showed `✅ Model checksum verified against expected SHA-256`, and `GET /ml/model-info` returned `"checksum_verified": true` right next to the unchanged `"taxonomy_compatible": false` — confirming the two signals are independent.
- **Mismatch startup (never crashes):** I started it again with a deliberately wrong expected SHA. The log showed the `❌ ... UNVERIFIED` error, but then `✅ YOLO ready` and `Application startup complete` — the backend stayed up and usable.

In [ ]:
# 1) Tests (run from the backend directory)
$ python3 -m pytest tests/
# -> 165 passed

# 2) Normal startup — expected SHA came from the baseline fallback (no env var set)
$ WALKBUDDY_MODEL_DIR=../../../ML_side/models python3 -m uvicorn main:app --port 8010
# startup log (relevant lines):
#   ✅ Model checksum verified against expected SHA-256
#   ✅ YOLO ready
#   Application startup complete.

$ curl -s localhost:8010/ml/model-info | python3 -m json.tool
# {
#   ...
#   "taxonomy_compatible": false,
#   "checksum_verified": true,      # <-- new field, independent of taxonomy
#   ...
# }

# 3) Mismatch startup — deliberately wrong expected value
$ WALKBUDDY_EXPECTED_MODEL_SHA256=0000000000000000000000000000000000000000000000000000000000000000 \
    WALKBUDDY_MODEL_DIR=../../../ML_side/models python3 -m uvicorn main:app --port 8011
# startup log (relevant lines):
#   ❌ Model checksum verification FAILED: ... integrity is UNVERIFIED ...
#   ✅ YOLO ready
#   Application startup complete.   # <-- backend stayed up; the mismatch never crashed it

## 6. What I learned / notes

- **Validating before coding paid off.** Because I checked the existing integration first (and searched the repo), I found a *real* gap — the hash was recorded but never verified — instead of re-building something that already existed. That felt like the point of the "validate" half of the task.
- **"Pure addition" is a useful discipline.** Keeping the change additive (never crash, don't touch `/vision` / `/ml/navigate`, don't replace the model) means it can't make the existing, working system worse. The worst case of my feature is that it *reports* a problem — it never *causes* one.
- **Keep signals separate.** `checksum_verified` (file integrity) and `taxonomy_compatible` (class labels) answer different questions, so I kept them as two independent fields. Seeing `checksum_verified: true` next to `taxonomy_compatible: false` on the live endpoint made that separation concrete.
- **Config beats hardcoding.** Sourcing the expected SHA from an env var with a committed-baseline fallback avoids a brittle hash literal in the code. One follow-up I'd recommend: in the **production deployment**, set `WALKBUDDY_EXPECTED_MODEL_SHA256` explicitly so the approved value is pinned at the deploy layer, not just read from a repo file that could change if the model is swapped later.

---

